# ARENA3 2단계 — S2A (이득 회귀 → 강화학습)

B′ 교사 자료(학습 8일 64판, 복원이 맞은 선택 지점 1,802개)로 먼저 가르친 뒤, G1과 같은 SMDP 강화학습으로 이어 학습합니다.
- **사전학습:** 각 신호가 M4보다 얼마나 나은지(B′ 롤아웃의 짝 이득 − 표준오차 1배)를 맞히도록 가르칩니다. 확신이 큰 지점에 무게를 더 둡니다.
- **강화학습:** G1과 같은 설정(듀얼링 DQN, M4 기준, 관측 119칸, 선택 지점끼리 잇기)입니다. 교사 손실(무게 0.2)을 학습 끝까지 함께 씁니다.
- 신경망이 보는 것은 **관측 119칸뿐**입니다. 날짜·미래 차량·롤아웃 결과는 정답(라벨)으로만 쓰고 입력에 넣지 않습니다.
- 예산은 G1과 같습니다. 난수마다 작업시간 20시간(합 60시간)이고, 사전학습 시간도 포함합니다.

1. 런타임을 **CPU(높은 RAM)**로 고르고 **런타임 → 모두 실행**을 누르세요. **S2B 노트북과 동시에** 돌리면 됩니다.
2. 순서: 참고 진단(약 2분) → 실행 전 점검(약 5분) → 학습 난수 3개 동시에(약 10~11시간) → 심판 평가 504회(약 2시간) → 참고 판정.
3. 끊기면 처음 네 칸을 실행한 뒤 멈춘 칸부터 다시 실행하세요(또는 **모두 실행**). 끝난 단계는 건너뛰고, 학습은 체크포인트에서 이어집니다.

In [ ]:
from google.colab import drive
from pathlib import Path
import os, sys, json

drive.mount("/content/drive")
DRIVE = Path("/content/drive/MyDrive/FutureTraffic")
ARCHIVE = DRIVE / "FutureTraffic_ARENA3_S2.zip"
A3 = DRIVE / "arena3"
S2 = A3 / "stage2"
DATA = A3 / "bprime_data"
WORKSPACE = Path("/content/FutureTraffic_ARENA3_S2")
assert ARCHIVE.is_file(), f"묶음이 없습니다: {ARCHIVE}"
assert (DATA / "experiment.json").exists(), f"B′ 교사 자료가 없습니다: {DATA}"
CPUS = os.cpu_count() or 1
VARIANT, NAME = "advantage", "S2A"
print("CPU 개수:", CPUS, "| 결과 폴더:", S2)

In [ ]:
# ZIP 전체의 파일 목록·SHA-256을 확인한 뒤 검증된 전용 추출기를 사용합니다.
import hashlib, stat, types, zipfile
from pathlib import PurePosixPath

with zipfile.ZipFile(ARCHIVE) as archive:
    names = archive.namelist()
    assert len(names) == len(set(names)) and names.count("manifest.json") == 1, "ZIP 파일 목록 오류"
    package_manifest = json.loads(archive.read("manifest.json"))
    assert package_manifest.get("kind") == "WEIGHT1_PACKAGE_V1", "검증할 수 없는 ZIP 형식입니다"
    assert package_manifest.get("experiment") == "ARENA3_S2_V1", "ARENA3 2단계 전용 ZIP이 아닙니다"
    assert package_manifest.get("source_models_included") is False, "모델·결과가 든 묶음은 쓰지 않습니다"
    expected = package_manifest["files"]
    assert expected and "manifest.json" not in expected and set(names) == set(expected) | {"manifest.json"}, "ZIP과 manifest 목록 불일치"
    for info in archive.infolist():
        name, path = info.filename, PurePosixPath(info.filename)
        assert path.parts and not path.is_absolute() and ".." not in path.parts and path.as_posix() == name, "잘못된 ZIP 경로"
        assert "\\" not in name and not info.is_dir() and not stat.S_ISLNK(info.external_attr >> 16), "잘못된 ZIP 파일 형식"
        if name != "manifest.json":
            assert hashlib.sha256(archive.read(name)).hexdigest() == expected[name], f"SHA-256 불일치: {name}"
    source_code = archive.read("src/explore_weight_package.py").decode("utf-8")

bootstrap = types.ModuleType("arena3_verified_package")
bootstrap.__file__ = str(WORKSPACE / "src/explore_weight_package.py")
exec(compile(source_code, bootstrap.__file__, "exec"), bootstrap.__dict__)
package_manifest = bootstrap.extract_verified(ARCHIVE, WORKSPACE)
print(f"검증·추출 완료: {len(package_manifest['files'])}개 파일 → {WORKSPACE} | 잠금 {package_manifest['lock_sha256'][:12]} "
      f"| 2단계 설정 {package_manifest['stage2_config_sha256'][:12]}")

In [ ]:
import subprocess, importlib.metadata

os.environ["FOCUSED_RULESET"] = "pet1"
os.environ["FOCUSED_NORTH_EXT"] = "240"
EXPECTED = {"numpy": "2.1.3", "torch": "2.11.0+cpu", "stable-baselines3": "2.9.0",
            "sumo-rl": "1.4.5", "eclipse-sumo": "1.27.1"}

def installed_versions():
    versions = {}
    for name in EXPECTED:
        try:
            versions[name] = importlib.metadata.version(name)
        except importlib.metadata.PackageNotFoundError:
            versions[name] = None
    return versions

before = installed_versions()
if before == EXPECTED:
    print("이전 실험과 패키지 버전이 같아 재설치를 건너뜁니다.")
else:
    print("이전 실험과 같은 패키지 버전을 준비합니다:", before)
    subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements.txt"], cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "numpy==2.1.3", "stable-baselines3==2.9.0", "eclipse-sumo==1.27.1"],
                   cwd=WORKSPACE, check=True)
    subprocess.run([sys.executable, "-m", "pip", "install", "torch==2.11.0+cpu", "--index-url", "https://download.pytorch.org/whl/cpu"],
                   cwd=WORKSPACE, check=True)
after = installed_versions()
assert after == EXPECTED, f"패키지 버전이 이전 실험과 다릅니다(평가 지문이 달라져 기준선과 비교할 수 없어 멈춥니다): {after}"
print("실행 패키지 검증 통과:", after)

In [ ]:
sys.path.insert(0, str(WORKSPACE / "src"))
import arena3_colab as colab
from IPython.display import display, Markdown

colab.run_live(["-m", "unittest", "discover", "-s", "tests", "-p", "test_arena3*.py"], WORKSPACE)
print("실행 전 검사(SUMO 없이) 통과")

In [ ]:
# 1) 참고 진단(SUMO 없음, 약 2분): 학습 날짜 하나를 통째로 빼고 나머지로 가르친 뒤, 뺀 날짜에서 관측 119칸만 보고
#    M4보다 나은 선택을 가려내는지 봅니다. 점수는 B′가 판단에 쓰지 않은 표본 미래로 매깁니다. 결과와 상관없이 다음 칸으로 갑니다.
OFFLINE = S2 / "offline"
if not (OFFLINE / "offline_report.md").exists():
    colab.run_live(["src/arena3_stage2_data.py", "offline", "--data", str(DATA), "--out", str(OFFLINE)], WORKSPACE)
display(Markdown((OFFLINE / "offline_report.md").read_text()))

In [ ]:
# 2) 실행 전 점검: 진짜 SUMO로 사전학습 + 한 묶음(에피소드 2개)을 점검용 폴더에서 학습합니다. 본 학습 폴더와 섞이지 않습니다.
SMOKE = S2 / f"{NAME}_smoke"
if (SMOKE / "smoke.json").exists():
    print("이전 점검이 통과해 건너뜁니다:", colab.read_json(SMOKE / "smoke.json")["episodes"], "에피소드")
else:
    colab.run_live(["src/arena3_stage2_train.py", "smoke", "--variant", VARIANT, "--seed", "42", "--data", str(DATA),
                    "--out", str(SMOKE), "--work", f"/content/{NAME}_smoke"], WORKSPACE)

In [ ]:
# 3) 학습: 난수 42·43·44를 동시에(각각 에피소드 2개씩 병렬). 난수마다 작업시간 20시간(사전학습 포함)에서 멈춥니다.
#    셀을 멈춰도 학습은 계속 돕니다. 런타임이 끊기면 처음 네 칸 → 이 칸을 다시 실행하면 체크포인트에서 이어집니다.
TRAIN = S2 / f"{NAME}_train"
LOGS = A3 / "logs" / NAME
jobs = {f"{NAME}_seed{s}": ["src/arena3_stage2_train.py", "train", "--variant", VARIANT, "--seed", str(s),
                             "--data", str(DATA), "--out", str(TRAIN), "--work", f"/content/{NAME}_{s}"]
        for s in (42, 43, 44)}

def describe(job):
    s = colab.read_json(TRAIN / f"seed_{job[-2:]}" / "status.json")
    return None if not s else (f"에피소드 {s['episodes']}, 선택 {s['choices']}, 버린 판 {s['discarded']}, "
                               f"작업시간 {s['cpu_hours']:.2f}/{s['budget_cpu_hours']:.0f}시간, 끝남 {s['finished']}")

pids = colab.launch(jobs, WORKSPACE, LOGS)
code = colab.watch(pids, LOGS, 300, describe)
assert code == 0, f"{NAME} 학습에서 오류가 났습니다. 위 로그를 저에게 보내 주세요."

In [ ]:
# 4) 심판 평가: 사전학습만 한 모델({NAME}_pre)과 강화학습까지 한 최종 모델({NAME})을 각각 3개 × 84상황 = 504회.
#    1단계와 같은 심판·같은 84상황입니다. 끊겨도 다시 실행하면 끝난 판은 건너뜁니다.
assert "colab" in globals(), "처음 네 칸(1~4)을 먼저 실행하세요. 런타임이 끊기면 변수와 작업 폴더가 사라집니다."
TRAIN = S2 / f"{NAME}_train"  # 학습 칸(3)을 건너뛰고 이 칸만 실행해도 되도록 여기서도 정한다
EVAL = S2 / "eval"
for name, stage in ((f"{NAME}_pre", "pretrained"), (NAME, "final")):
    models = [TRAIN / f"seed_{s}" / stage for s in (42, 43, 44)]
    assert all((m / "model.pt").exists() for m in models), f"{stage} 모델이 아직 없습니다. 3)을 다시 실행하세요."
    colab.run_live(["src/arena3_stage2_eval.py", "evaluate", "--name", name, "--models", *map(str, models),
                    "--out", str(EVAL / name), "--work", f"/content/{name}_eval", "--workers", str(CPUS)], WORKSPACE)
print(f"{NAME} 평가 끝. S2B 노트북도 끝나면 S2C(판정·확인)를 실행하세요.")

In [ ]:
# 5) 지금까지의 판정(참고): S2B가 아직이면 '아직 평가 중'으로 나옵니다. 최종 판정과 C1·C2 확인은 S2C 노트북에서 합니다.
BASE = A3 / "referee" / "baseline"
CANDIDATES = [f"W4={BASE}", f"BPRIME={A3 / 'referee' / 'bprime'}", f"G1={A3 / 'referee' / 'G1'}"]
CANDIDATES += [f"{n}={S2 / 'eval' / n}" for n in ("S2A_pre", "S2A", "S2B_pre", "S2B")]
judge_args = ["src/arena3_stage2_eval.py", "judge", "--baseline", str(BASE), "--out", str(S2 / "judge")]
for c in CANDIDATES:
    judge_args += ["--candidate", c]
colab.run_live(judge_args, WORKSPACE, check=False)